In [14]:
import numpy as np
from enum import Enum
from scipy import stats

In [15]:
class GaussParam(Enum):
    MU = 0
    SIGMA = 1

In [16]:
def gauss_conf(
    xs: np.ndarray[np.float64],
    prob: float,
    param: GaussParam,
    mu: float | None = None,
    sigma: float | None = None,
) -> tuple[float, float]:
    n = xs.size
    q = 1.0 - (1.0 - prob) * 0.5

    match param:
        case GaussParam.MU:
            mean = xs.mean()

            if sigma is None:
                se = xs.std(ddof=1) / np.sqrt(n)
                crit = stats.t.ppf(q, df=n-1)
            else:
                se = sigma / np.sqrt(n)
                crit = stats.norm.ppf(q)

            offset = crit * se
            return (mean - offset, mean + offset)

        case GaussParam.SIGMA:
            df = n - 1
            s2 = xs.var(ddof=1)

            if mu is None:
                chi2_lo = stats.chi2.ppf(1 - q, df)
                chi2_hi = stats.chi2.ppf(q, df)
            else:
                df = n
                s2 = np.mean((xs - mu) ** 2)
                
                chi2_lo = stats.chi2.ppf(1 - q, df)
                chi2_hi = stats.chi2.ppf(q, df)

            var_lo = df * s2 / chi2_hi
            var_hi = df * s2 / chi2_lo

            return (np.sqrt(var_lo), np.sqrt(var_hi))

In [17]:
def solve(n: int, prob=0.95):
    xs = np.random.normal(size=n)

    mu_lo, mu_hi = gauss_conf(xs, prob, GaussParam.MU)
    mu_lo_k, mu_hi_k = gauss_conf(xs, prob, GaussParam.MU, sigma=1.0)
    sig_lo, sig_hi = gauss_conf(xs, prob, GaussParam.SIGMA)
    sig_lo_k, sig_hi_k = gauss_conf(xs, prob, GaussParam.SIGMA, mu=0.0)

    print(f"n = {n}")
    print(f"  μ (σ неизв.): [{mu_lo:+.4f}, {mu_hi:+.4f}]  ширина = {mu_hi - mu_lo:.4f}")
    print(f"  μ (σ изв.):   [{mu_lo_k:+.4f}, {mu_hi_k:+.4f}]  ширина = {mu_hi_k - mu_lo_k:.4f}")
    print(f"  σ (μ неизв.): [{sig_lo:.4f}, {sig_hi:.4f}]  ширина = {sig_hi - sig_lo:.4f}")
    print(f"  σ (μ изв.):   [{sig_lo_k:.4f}, {sig_hi_k:.4f}]  ширина = {sig_hi_k - sig_lo_k:.4f}")
    print()

In [18]:
for n in (10, 100, 1000, 10000):
    solve(n)

n = 10
  μ (σ неизв.): [-1.6301, +0.2364]  ширина = 1.8665
  μ (σ изв.):   [-1.3166, -0.0771]  ширина = 1.2396
  σ (μ неизв.): [0.8973, 2.3816]  ширина = 1.4843
  σ (μ изв.):   [0.9924, 2.4926]  ширина = 1.5002

n = 100
  μ (σ неизв.): [-0.1580, +0.3050]  ширина = 0.4630
  μ (σ изв.):   [-0.1225, +0.2695]  ширина = 0.3920
  σ (μ неизв.): [1.0245, 1.3555]  ширина = 0.3310
  σ (μ изв.):   [1.0220, 1.3503]  ширина = 0.3283

n = 1000
  μ (σ неизв.): [-0.0465, +0.0728]  ширина = 0.1194
  μ (σ изв.):   [-0.0488, +0.0751]  ширина = 0.1240
  σ (μ неизв.): [0.9214, 1.0059]  ширина = 0.0845
  σ (μ изв.):   [0.9211, 1.0055]  ширина = 0.0844

n = 10000
  μ (σ неизв.): [-0.0294, +0.0099]  ширина = 0.0392
  μ (σ изв.):   [-0.0293, +0.0099]  ширина = 0.0392
  σ (μ неизв.): [0.9867, 1.0145]  ширина = 0.0277
  σ (μ изв.):   [0.9867, 1.0145]  ширина = 0.0277



In [19]:

def coverage(n: int, N: int, prob: float = 0.95, seed: int = 0):
    rng = np.random.default_rng(seed)
    true_mu, true_sigma = 0.0, 1.0

    hits = {
        "mu_unknown": 0,
        "mu_known": 0,
        "sig_unknown": 0,
        "sig_known": 0,
    }

    for _ in range(N):
        xs = rng.normal(true_mu, true_sigma, size=n)

        lo, hi = gauss_conf(xs, prob, GaussParam.MU)
        hits["mu_unknown"] += lo <= true_mu <= hi

        lo, hi = gauss_conf(
            xs, prob, GaussParam.MU, sigma=true_sigma
        )
        hits["mu_known"] += lo <= true_mu <= hi

        lo, hi = gauss_conf(xs, prob, GaussParam.SIGMA)
        hits["sig_unknown"] += lo <= true_sigma <= hi

        lo, hi = gauss_conf(
            xs, prob, GaussParam.SIGMA, mu=true_mu
        )
        hits["sig_known"] += lo <= true_sigma <= hi

    return {key: value / N for key, value in hits.items()}

In [20]:
prob = 0.95

names = {
    "mu_unknown": "μ (σ неизвестна)",
    "mu_known": "μ (σ известна)",
    "sig_unknown": "σ (μ неизвестно)",
    "sig_known": "σ (μ известно)",
}

N = 200

for n in (10, 100, 1000, 10000):
    freq = coverage(n, N, prob=prob, seed=n)

    print(f"n = {n}, N = {N}")

    for key, name in names.items():
        print(f"  {name}: {freq[key]:.4f}")
        
    print()

N = 20000

for n in (10, 100, 1000, 10000):
    freq = coverage(n, N, prob=prob, seed=n)

    print(f"n = {n}, N = {N}")

    for key, name in names.items():
        print(f"  {name}: {freq[key]:.4f}")
        
    print()

n = 10, N = 200
  μ (σ неизвестна): 0.9450
  μ (σ известна): 0.9550
  σ (μ неизвестно): 0.9700
  σ (μ известно): 0.9350

n = 100, N = 200
  μ (σ неизвестна): 0.9150
  μ (σ известна): 0.9200
  σ (μ неизвестно): 0.9800
  σ (μ известно): 0.9850

n = 1000, N = 200
  μ (σ неизвестна): 0.9700
  μ (σ известна): 0.9750
  σ (μ неизвестно): 0.9350
  σ (μ известно): 0.9350

n = 10000, N = 200
  μ (σ неизвестна): 0.9500
  μ (σ известна): 0.9500
  σ (μ неизвестно): 0.9500
  σ (μ известно): 0.9500

n = 10, N = 20000
  μ (σ неизвестна): 0.9520
  μ (σ известна): 0.9517
  σ (μ неизвестно): 0.9504
  σ (μ известно): 0.9505

n = 100, N = 20000
  μ (σ неизвестна): 0.9507
  μ (σ известна): 0.9497
  σ (μ неизвестно): 0.9511
  σ (μ известно): 0.9510

n = 1000, N = 20000
  μ (σ неизвестна): 0.9495
  μ (σ известна): 0.9496
  σ (μ неизвестно): 0.9488
  σ (μ известно): 0.9489

n = 10000, N = 20000
  μ (σ неизвестна): 0.9493
  μ (σ известна): 0.9495
  σ (μ неизвестно): 0.9500
  σ (μ известно): 0.9500



In [21]:
class BernParam(Enum):
    P = 0
    VAR = 1

In [22]:
def bern_conf(
    xs: np.ndarray,
    prob: float,
    param: BernParam,
    p: float | None = None,
) -> tuple[float, float]:
    n = xs.size
    sn = int(xs.sum())

    q = 1.0 - (1.0 - prob) * 0.5
    z = stats.norm.ppf(q)
    z2 = z * z

    match param:
        case BernParam.P:
            center = (2 * sn + z2) / (2 * (n + z2))
            half = z * np.sqrt(4 * sn * (n - sn) / n + z2) / (2 * (n + z2))

            p_lo = max(0.0, center - half)
            p_hi = min(1.0, center + half)

            return (p_lo, p_hi)

        case BernParam.VAR:
            if p is None:
                p_lo, p_hi = bern_conf(xs, prob, BernParam.P)
            else:
                p_lo = p_hi = p

            h = lambda x: x * (1.0 - x)

            if p_hi <= 0.5:
                d_lo, d_hi = h(p_lo), h(p_hi)
            elif p_lo >= 0.5:
                d_lo, d_hi = h(p_hi), h(p_lo)
            else:
                d_lo, d_hi = min(h(p_lo), h(p_hi)), 0.25

            return (d_lo, d_hi)

In [23]:
def solve(n: int, p_true: float = 0.4, prob: float = 0.95, seed: int = 0):
    rng = np.random.default_rng(seed)
    xs = (rng.random(n) < p_true).astype(np.float64)

    p_lo, p_hi = bern_conf(xs, prob, BernParam.P)
    d_lo, d_hi = bern_conf(xs, prob, BernParam.VAR)
    d_lo_k, d_hi_k = bern_conf(xs, prob, BernParam.VAR, p=p_true)

    print(f"n = {n}")
    print(f"  p (неизв.)      : [{p_lo:.4f}, {p_hi:.4f}]  ширина = {p_hi - p_lo:.4f}")
    print(f"  p(1-p) (неизв.) : [{d_lo:.4f}, {d_hi:.4f}]  ширина = {d_hi - d_lo:.4f}")
    print(f"  p(1-p) (p изв.) : [{d_lo_k:.4f}, {d_hi_k:.4f}]  ширина = {d_hi_k - d_lo_k:.4f}")
    print()

In [24]:
for n in (10, 100, 1000, 10000):
    solve(n, p_true=0.4, prob=0.95, seed=n)

n = 10
  p (неизв.)      : [0.1078, 0.6032]  ширина = 0.4954
  p(1-p) (неизв.) : [0.0962, 0.2500]  ширина = 0.1538
  p(1-p) (p изв.) : [0.2400, 0.2400]  ширина = 0.0000

n = 100
  p (неизв.)      : [0.2367, 0.4166]  ширина = 0.1799
  p(1-p) (неизв.) : [0.1807, 0.2430]  ширина = 0.0624
  p(1-p) (p изв.) : [0.2400, 0.2400]  ширина = 0.0000

n = 1000
  p (неизв.)      : [0.3799, 0.4408]  ширина = 0.0609
  p(1-p) (неизв.) : [0.2356, 0.2465]  ширина = 0.0109
  p(1-p) (p изв.) : [0.2400, 0.2400]  ширина = 0.0000

n = 10000
  p (неизв.)      : [0.3793, 0.3984]  ширина = 0.0191
  p(1-p) (неизв.) : [0.2354, 0.2397]  ширина = 0.0042
  p(1-p) (p изв.) : [0.2400, 0.2400]  ширина = 0.0000



In [25]:
def coverage(n: int, N: int, p_true: float = 0.4,
             prob: float = 0.95, seed: int = 0):
    rng = np.random.default_rng(seed)

    hits = {
        "p_unknown": 0,
        "var_unknown": 0,
        "var_known": 0,
    }

    true_var = p_true * (1.0 - p_true)

    for _ in range(N):
        xs = (rng.random(n) < p_true).astype(np.float64)

        lo, hi = bern_conf(xs, prob, BernParam.P)
        hits["p_unknown"] += lo <= p_true <= hi

        lo, hi = bern_conf(xs, prob, BernParam.VAR)
        hits["var_unknown"] += lo <= true_var <= hi

        lo, hi = bern_conf(xs, prob, BernParam.VAR, p=p_true)
        hits["var_known"] += lo <= true_var <= hi

    return {key: value / N for key, value in hits.items()}

In [26]:
prob = 0.95

names = {
    "p_unknown":   "p (неизв.)",
    "var_unknown": "p(1-p) (p неизв.)",
    "var_known":   "p(1-p) (p изв.)",
}

N = 200

for n in (10, 100, 1000, 10000):
    freq = coverage(n, N, p_true=0.4, prob=prob, seed=n)

    print(f"n = {n}, N = {N}")

    for key, name in names.items():
        print(f"  {name}: {freq[key]:.4f}")

    print()

N = 20000

for n in (10, 100, 1000, 10000):
    freq = coverage(n, N, p_true=0.4, prob=prob, seed=n)

    print(f"n = {n}, N = {N}")

    for key, name in names.items():
        print(f"  {name}: {freq[key]:.4f}")

    print()

n = 10, N = 200
  p (неизв.): 0.9850
  p(1-p) (p неизв.): 0.9950
  p(1-p) (p изв.): 1.0000

n = 100, N = 200
  p (неизв.): 0.9550
  p(1-p) (p неизв.): 0.9750
  p(1-p) (p изв.): 1.0000

n = 1000, N = 200
  p (неизв.): 0.9300
  p(1-p) (p неизв.): 0.9300
  p(1-p) (p изв.): 1.0000

n = 10000, N = 200
  p (неизв.): 0.9400
  p(1-p) (p неизв.): 0.9400
  p(1-p) (p изв.): 1.0000

n = 10, N = 20000
  p (неизв.): 0.9803
  p(1-p) (p неизв.): 0.9936
  p(1-p) (p изв.): 1.0000

n = 100, N = 20000
  p (неизв.): 0.9478
  p(1-p) (p неизв.): 0.9644
  p(1-p) (p изв.): 1.0000

n = 1000, N = 20000
  p (неизв.): 0.9482
  p(1-p) (p неизв.): 0.9482
  p(1-p) (p изв.): 1.0000

n = 10000, N = 20000
  p (неизв.): 0.9506
  p(1-p) (p неизв.): 0.9506
  p(1-p) (p изв.): 1.0000

